# Add variables to the TransferHub format

In [ ]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path

# Run in parent dir
cwd = Path.cwd().resolve()
if cwd.name == "notebooks":
    os.chdir(cwd.parent)

import pandas as pd

data_dir = Path('data')
IMAGE_TYPES = ['rt_leftcentral', 'rt_leftnasal', 'rt_rightcentral', 'rt_rightnasal']

In [ ]:
# Load IDs and image existence
df_images = pd.read_csv(data_dir / "all_images.csv", dtype={'ID': str})
df_images.head()

# Start with a df only with IDs
df = pd.DataFrame(df_images['ID'])
df.rename(columns={'ID': 'Proband'}, inplace=True)

In [ ]:
MISSING_RAW, EVAL_FAILED = -1, -4

def add_variable(df, df_variable, variable_name, images_exist):
    """Add one variable and its missing-code column (<variable_name>_m) to df."""

    # Merge variable with df
    df = pd.merge(df, df_variable[['Proband', variable_name]], how='left', on='Proband')

    has_image = df['Proband'].isin(images_exist)
    no_value = df[variable_name].isna()

    # Missing codes go in a separate column; value column stays empty (null) where missing
    missing = f'{variable_name}_m'
    df[missing] = pd.Series(pd.NA, index=df.index, dtype='Int64')

    # Images that do not exist
    df.loc[~has_image, missing] = MISSING_RAW

    # Images that exist but have no value (evaluation failed)
    df.loc[has_image & no_value, missing] = EVAL_FAILED

    assert not (~has_image & ~no_value).any(), f'{variable_name}: values for images that do not exist'

    return df

## Add quality gradings

In [ ]:
# Quality suffixes for each model
SUFFIX = {'rt_leftcentral': 'lc', 'rt_leftnasal': 'ln',
          'rt_rightcentral': 'rc', 'rt_rightnasal': 'rn'}

def add_model(df, model_results, variables, df_images):
    """Add all variables from one model to df."""
    for image_type, suffix in SUFFIX.items():
        for i in range(1, 4):
            key = f'{image_type}_{i}'
            images_exist = df_images.loc[df_images[key], 'ID']
            for column, (name, transform) in variables.items():
                variable_name = f'u590_{name}_{suffix}{i}'
                df_variable = model_results[key][['Proband', column]].rename(columns={column: variable_name})
                if transform is not None:
                    df_variable[variable_name] = transform(df_variable[variable_name])
                df = add_variable(df, df_variable, variable_name, images_exist)
    return df

In [ ]:
# Load results from all quality filters
fit, retiratey, automorph, vascx = {}, {}, {}, {}
quality_dir = data_dir / "quality"

for image_type in IMAGE_TYPES:
    for i in range(1, 4):
        quality_file = quality_dir.joinpath('fundus_image_toolbox', f'{image_type}_{str(i)}.csv')
        fit[f'{image_type}_{i}'] = pd.read_csv(quality_file, dtype={'ID': str})
        fit[f'{image_type}_{i}'].rename(columns={'ID': 'Proband'}, inplace=True)

        quality_file = quality_dir.joinpath('retiratey', f'{image_type}_{str(i)}.csv')
        retiratey[f'{image_type}_{i}'] = pd.read_csv(quality_file, dtype={'ID': str})
        retiratey[f'{image_type}_{i}'].rename(columns={'ID': 'Proband'}, inplace=True)

        quality_file = quality_dir.joinpath('automorph', f'{image_type}_{str(i)}.csv')
        automorph[f'{image_type}_{i}'] = pd.read_csv(quality_file, dtype={'ID': str})
        automorph[f'{image_type}_{i}'].rename(columns={'ID': 'Proband'}, inplace=True)

        quality_file = quality_dir.joinpath('vascx', f'{image_type}_{str(i)}.csv')
        vascx[f'{image_type}_{i}'] = pd.read_csv(quality_file, dtype={'ID': str})
        vascx[f'{image_type}_{i}'].rename(columns={'ID': 'Proband'}, inplace=True)

### FIT

In [ ]:
# Add FIT variables to df
FIT_VARIABLES = {
    # results column: (base name, transform)
    'confidence': ('qfit_conf', None),
    'label': ('qfit_cls', lambda v: v.map({0: 1, 1: 2}).astype('Int64')),  # 1=Ungradable, 2=Gradable
}

df = add_model(df, fit, FIT_VARIABLES, df_images)

### AutoMorph

In [ ]:
# Add AutoMorph variables to df
AUTOMORPH_VARIABLES = {
    # results column: (base name, transform)
    'prediction_label': ('qam_cls', lambda v: v.map({'Good': 1, 'Usable': 2, 'Reject': 3}).astype('Int64')),  # 1=Good, 2=Usable, 3=Reject
    'softmax_good': ('qam_pgd', None),
    'softmax_usable': ('qam_pus', None),
    'softmax_bad': ('qam_prj', None),
    'good_sd': ('qam_sdgd', None),
    'usable_sd': ('qam_sdus', None),
    'bad_sd': ('qam_sdrj', None),
}

df = add_model(df, automorph, AUTOMORPH_VARIABLES, df_images)

### VascX

In [ ]:
# Add VascX variables to df
VASCX_VARIABLES = {
    # results column: (base name, transform)
    'prediction_label': ('qvx_cls', lambda v: v.map({'Good': 1, 'Usable': 2, 'Reject': 3}).astype('Int64')),  # 1=Good, 2=Usable, 3=Reject
    'q1': ('qvx_lgd', None),
    'q2': ('qvx_lus', None),
    'q3': ('qvx_lrj', None),
}

df = add_model(df, vascx, VASCX_VARIABLES, df_images)

In [ ]:
df.head()

### RetiRatey

In [ ]:
# Add RetiRatey variables to df
RETIRATEY_VARIABLES = {
    # results column: (base name, transform)
    'macula': ('qrr_mac', None),
    'disc': ('qrr_disc', None),
    'vessels': ('qrr_ves', None),
}

df = add_model(df, retiratey, RETIRATEY_VARIABLES, df_images)

In [ ]:
df.head()

## Morphological measurements from AutoMorph

In [ ]:
metadata_file = data_dir / "metadata_590.xlsx"
metadata = pd.read_excel(metadata_file, sheet_name='Datenelemente', dtype=str)

# NAKO variable name to AutoMorph variable name mapping
morph_meta = metadata[metadata['Name'].str.startswith('u590_m_') & ~metadata['Name'].str.endswith('_m')]
MORPH_VARIABLES = dict(zip(
    morph_meta['Name'],
    morph_meta['Beschreibung'].str.extract(r'AutoMorph variable: (\w+)\.?$')[0],
))

# Load results from AutoMorph
SUFFIX_AM = {'l': 'rt_leftcentral', 'r': 'rt_rightcentral'}
am_dir = data_dir / "automorph_features"
automorph_features = {}
for image_type in SUFFIX_AM.values():
    am_file = am_dir / f'{image_type}.csv'
    automorph_features[f'{image_type}'] = pd.read_csv(am_file, dtype={'Name': str})
    automorph_features[f'{image_type}'].rename(columns={'Name': 'Proband'}, inplace=True)
    automorph_features[f'{image_type}']['Proband'] = automorph_features[f'{image_type}']['Proband'].map(lambda x: x.split('.')[0])

In [ ]:
for variable_name, column in MORPH_VARIABLES.items():
    laterality = SUFFIX_AM[variable_name[-1]]
    images_exist = df_images.loc[df_images[laterality + '_1'], 'ID']
    df_variable = automorph_features[laterality][['Proband', column]].rename(columns={column: variable_name})
    df = add_variable(df, df_variable, variable_name, images_exist)

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.to_csv(data_dir / "nako590_derived_variables.csv", sep=';', index=False, na_rep='', lineterminator='\n',  float_format='%.10f', encoding='utf-8')